
# ArchiveLens — NLP-Based Historical Document Intelligence System

### Mini Project — Natural Language Processing

**Objective:**  
Build a simple NLP system that automatically downloads a historical document from the internet and analyzes it using:
- Text preprocessing
- Stop-word removal and lemmatization
- N-Gram analysis
- Named Entity Recognition (NER)
- Text similarity using TF-IDF and Cosine Similarity

**Dataset:** *The Autobiography of Benjamin Franklin* from Project Gutenberg.  
The notebook downloads the text automatically, so **no file upload or manual data entry is required**.

Project Gutenberg provides the book as a free plain-text eBook. The work is an unfinished memoir written between 1771 and 1790.  
Source: https://www.gutenberg.org/ebooks/148


In [1]:

# 1. Install the small set of libraries we need
!pip -q install nltk spacy scikit-learn
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:

# 2. Import libraries
import re
import requests
import nltk
import spacy

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nlp = spacy.load("en_core_web_sm")

print("Libraries loaded successfully.")


Libraries loaded successfully.


In [3]:

# 3. Download the historical document automatically
URL = "https://www.gutenberg.org/files/148/148-0.txt"

response = requests.get(URL)
response.raise_for_status()

raw_text = response.text

print("Document downloaded successfully.")
print("Characters:", len(raw_text))
print("\nFirst 500 characters:\n")
print(raw_text[:500])


Document downloaded successfully.
Characters: 384915

First 500 characters:

﻿﻿*** START OF THE PROJECT GUTENBERG EBOOK 148 ***
THE AUTOBIOGRAPHY OF BENJAMIN FRANKLIN

The Harvard Classics

WITH INTRODUCTION AND NOTES

EDITED BY

CHARLES W ELIOT LLD

P F COLLIER & SON COMPANY
NEW YORK
1909




Navigation

    Letter from Mr. Abel James.
    Publishes the first number of "Poor Richard's Almanac.
    Proposes a Plan of Union for the colonies
    Chief events in Franklin's life.

INTRODUCTORY NOTE

Benjamin Franklin was born in Milk Street, Boston


In [4]:

# 4. Basic cleaning
# Remove the Project Gutenberg header/footer and unnecessary whitespace.

start = raw_text.find("*** START OF THE PROJECT GUTENBERG EBOOK")
end = raw_text.find("*** END OF THE PROJECT GUTENBERG EBOOK")

if start != -1:
    raw_text = raw_text[start:].split("\n", 1)[-1]

if end != -1:
    raw_text = raw_text[:end]

clean_text = re.sub(r"\s+", " ", raw_text).strip()

print("Cleaned text length:", len(clean_text))
print("\nSample:\n")
print(clean_text[:800])


Cleaned text length: 376971

Sample:

THE AUTOBIOGRAPHY OF BENJAMIN FRANKLIN The Harvard Classics WITH INTRODUCTION AND NOTES EDITED BY CHARLES W ELIOT LLD P F COLLIER & SON COMPANY NEW YORK 1909 Navigation Letter from Mr. Abel James. Publishes the first number of "Poor Richard's Almanac. Proposes a Plan of Union for the colonies Chief events in Franklin's life. INTRODUCTORY NOTE Benjamin Franklin was born in Milk Street, Boston, on January 6, 1706. His father, Josiah Franklin, was a tallow chandler who married twice, and of his seventeen children Benjamin was the youngest son. His schooling ended at ten, and at twelve he was bound apprentice to his brother James, a printer, who published the "New England Courant." To this journal he became a contributor, and later was for a time its nominal editor. But the brothers quarreled,



## 5. Text Preprocessing

The following steps remove punctuation and common stop words and convert words into simpler lemma forms.


In [5]:

# 5. Tokenization, filtration, stop-word removal and lemmatization

tokens = re.findall(r"\b[a-zA-Z]+\b", clean_text.lower())

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

processed_tokens = [
    lemmatizer.lemmatize(word)
    for word in tokens
    if word not in stop_words and len(word) > 2
]

print("Original token count:", len(tokens))
print("Processed token count:", len(processed_tokens))
print("\nFirst 50 processed tokens:")
print(processed_tokens[:50])


Original token count: 68330
Processed token count: 30521

First 50 processed tokens:
['autobiography', 'benjamin', 'franklin', 'harvard', 'classic', 'introduction', 'note', 'edited', 'charles', 'eliot', 'lld', 'collier', 'son', 'company', 'new', 'york', 'navigation', 'letter', 'abel', 'james', 'publishes', 'first', 'number', 'poor', 'richard', 'almanac', 'proposes', 'plan', 'union', 'colony', 'chief', 'event', 'franklin', 'life', 'introductory', 'note', 'benjamin', 'franklin', 'born', 'milk', 'street', 'boston', 'january', 'father', 'josiah', 'franklin', 'tallow', 'chandler', 'married', 'twice']



## 6. N-Gram Analysis

N-Grams help identify frequently occurring word sequences in the historical document.


In [7]:
processed_text = " ".join(processed_tokens)

vectorizer = CountVectorizer(ngram_range=(2, 3), min_df=1)
ngram_matrix = vectorizer.fit_transform([processed_text])

frequencies = ngram_matrix.toarray()[0]
ngrams = vectorizer.get_feature_names_out()

top_ngrams = sorted(
    zip(ngrams, frequencies),
    key=lambda x: x[1],
    reverse=True
)[:15]

print("Top 15 frequent Bigrams/Trigrams:")
for phrase, count in top_ngrams:
    print(f"{phrase:45} -> {count}")

Top 15 frequent Bigrams/Trigrams:
printing house                                -> 33
new york                                      -> 29
thousand pound                                -> 19
good deal                                     -> 18
young man                                     -> 16
new england                                   -> 15
one another                                   -> 14
would give                                    -> 11
hundred pound                                 -> 10
let know                                      -> 10
one hundred                                   -> 10
public affair                                 -> 10
every thing                                   -> 9
many year                                     -> 9
benjamin franklin                             -> 8



## 7. Named Entity Recognition

NER identifies entities such as people, places, organizations and dates from the historical text.


In [8]:

# 7. Extract named entities

# Process a manageable portion for quick execution.
sample_text = clean_text[:30000]

doc = nlp(sample_text)

entities = []
for ent in doc.ents:
    entities.append((ent.text, ent.label_))

entity_counts = Counter(entities)

print("Top named entities:")
for (entity, label), count in entity_counts.most_common(20):
    print(f"{entity:35} | {label:10} | {count}")


Top named entities:
first                               | ORDINAL    | 8
Boston                              | GPE        | 8
Benjamin                            | PERSON     | 7
one                                 | CARDINAL   | 6
London                              | GPE        | 5
England                             | GPE        | 5
Ecton                               | GPE        | 5
second                              | ORDINAL    | 4
Thomas                              | PERSON     | 4
John                                | PERSON     | 4
two                                 | CARDINAL   | 4
New England                         | LOC        | 4
ten                                 | CARDINAL   | 3
James                               | PERSON     | 3
Philadelphia                        | GPE        | 3
France                              | GPE        | 3
America                             | GPE        | 3
four                                | CARDINAL   | 3
Josiah                    


## 8. Real-Time Text Similarity

To demonstrate document similarity without asking the user to enter anything manually, the historical document is divided into sections. TF-IDF and cosine similarity are then used to find which sections are most similar.


In [9]:

# 8. Split the document into automatic sections

sentences = re.split(r"(?<=[.!?])\s+", clean_text)

# Create 8 automatic chunks
chunk_size = max(1, len(sentences) // 8)
chunks = [
    " ".join(sentences[i:i + chunk_size])
    for i in range(0, len(sentences), chunk_size)
]

chunks = chunks[:8]

tfidf = TfidfVectorizer(stop_words="english")
matrix = tfidf.fit_transform(chunks)

similarity_matrix = cosine_similarity(matrix)

print("Similarity between the automatically created document sections:")
print(similarity_matrix.round(2))


Similarity between the automatically created document sections:
[[1.   0.42 0.43 0.44 0.41 0.32 0.26 0.31]
 [0.42 1.   0.5  0.41 0.37 0.36 0.3  0.31]
 [0.43 0.5  1.   0.48 0.43 0.36 0.28 0.31]
 [0.44 0.41 0.48 1.   0.51 0.38 0.3  0.29]
 [0.41 0.37 0.43 0.51 1.   0.41 0.3  0.3 ]
 [0.32 0.36 0.36 0.38 0.41 1.   0.36 0.37]
 [0.26 0.3  0.28 0.3  0.3  0.36 1.   0.33]
 [0.31 0.31 0.31 0.29 0.3  0.37 0.33 1.  ]]


In [10]:

# 9. Display the most similar pair of sections

best_score = -1
best_pair = None

for i in range(len(chunks)):
    for j in range(i + 1, len(chunks)):
        if similarity_matrix[i, j] > best_score:
            best_score = similarity_matrix[i, j]
            best_pair = (i + 1, j + 1)

print("Most similar sections:", best_pair)
print("Cosine similarity:", round(best_score, 3))

print("\nSection", best_pair[0], "preview:")
print(chunks[best_pair[0] - 1][:500])

print("\nSection", best_pair[1], "preview:")
print(chunks[best_pair[1] - 1][:500])


Most similar sections: (4, 5)
Cosine similarity: 0.511

Section 4 preview:
But now another difficulty came upon me which I had never the least reason to expect. Mr. Meredith's father, who was to have paid for our printing-house, according to the expectations given me, was able to advance only one hundred pounds currency, which had been paid; and a hundred more was due to the merchant, who grew impatient, and su'd us all. We gave bail, but saw that, if the money could not be rais'd in time, the suit must soon come to a judgment and execution, and our hopeful prospects m

Section 5 preview:
Form of the pages. TEMPERANCE. eat not to dulness; drink not to elevation. S. M. T. W. T. F. S. T. S. • • • • O. •• • • • • • R. • • F. • • I. • S. J. M. C. T. C. H. I determined to give a week's strict attention to each of the virtues successively. Thus, in the first week, my great guard was to avoid every the least offence against Temperance, leaving the other virtues to their ordinary chance, only 


## 10. Simple ArchiveLens Summary

The system has automatically:
1. Downloaded a historical document.
2. Cleaned and tokenized the text.
3. Removed stop words and performed lemmatization.
4. Found frequent word sequences using N-Grams.
5. Extracted named entities using NER.
6. Compared automatically generated document sections using TF-IDF and cosine similarity.

No dataset upload or manual text input is required.


In [11]:

# 10. Final project statistics

print("========== ARCHIVELENS RESULTS ==========")
print("Document: The Autobiography of Benjamin Franklin")
print("Total characters:", len(clean_text))
print("Total original tokens:", len(tokens))
print("Total processed tokens:", len(processed_tokens))
print("Document sections compared:", len(chunks))
print("Unique named entities found:", len(entity_counts))
print("Highest N-Gram frequency:", top_ngrams[0][1] if top_ngrams else 0)
print("Highest section similarity:", round(best_score, 3))
print("==========================================")


========== ARCHIVELENS RESULTS ==========
Document: The Autobiography of Benjamin Franklin
Total characters: 376971
Total original tokens: 68330
Total processed tokens: 30521
Document sections compared: 8
Unique named entities found: 197
Highest N-Gram frequency: 33
Highest section similarity: 0.511



## Conclusion

ArchiveLens demonstrates how NLP can be applied to historical documents without requiring manual dataset preparation. The system uses basic preprocessing, N-Gram analysis, Named Entity Recognition and text similarity to convert an unstructured historical document into useful structured information.

**Data source:** Project Gutenberg — *The Autobiography of Benjamin Franklin*  
https://www.gutenberg.org/ebooks/148
